In [1]:
!pip install ultralytics

^C


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
easyocr 1.7.2 requires pyclipper, which is not installed.
easyocr 1.7.2 requires python-bidi, which is not installed.
easyocr 1.7.2 requires Shapely, which is not installed.


  Using cached sympy-1.13.1-py3-none-any.whl.metadata (12 kB)
   ---------------------------------------- 0.0/977.1 kB ? eta -:--:--
   ---------------------------------------- 0.0/977.1 kB ? eta -:--:--
   ---------- ----------------------------- 262.1/977.1 kB ? eta -:--:--
   -------------------- ----------------- 524.3/977.1 kB 885.6 kB/s eta 0:00:01
   -------------------- ----------------- 524.3/977.1 kB 885.6 kB/s eta 0:00:01
   ------------------------------ ------- 786.4/977.1 kB 884.1 kB/s eta 0:00:01
   -------------------------------------- 977.1/977.1 kB 880.5 kB/s eta 0:00:00
   ---------------------------------------- 0.0/204.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/204.1 MB ? eta -:--:--
   ---------------------------------------- 0.3/204.1 MB ? eta -:--:--
   ---------------------------------------- 0.5/204.1 MB 932.9 kB/s eta 0:03:39
   ---------------------------------------- 0.5/204.1 MB 932.9 kB/s eta 0:03:39
   ---------------------------

In [2]:
!git clone https://github.com/ultralytics/ultralytics.git

fatal: destination path 'ultralytics' already exists and is not an empty directory.


In [3]:
!git clone https://github.com/ultralytics/yolov5.git

fatal: destination path 'yolov5' already exists and is not an empty directory.


In [ ]:
!del output_message.mp3##done

In [ ]:
!pip install PyAudio

In [ ]:
!cd path_to_download_directory
!pip install filename.whl
!pip install PyAudio-0.2.11-cp39-cp39-win_amd64.whl

In [ ]:
!pip install roboflow

In [ ]:
!sudo apt-get install libgtk2.0-dev pkg-config


In [ ]:
!cd yolov5


In [ ]:
# le temps qui sépare entre la lecture des objets détectés en chaque frmae est environ 5 secondes
# seul les objects critiques sont lus dans l'alert (dans ce cas juste les personnes) 

In [ ]:
import cv2
from ultralytics import YOLO
from gtts import gTTS
from playsound import playsound
import tempfile
import os
import time
from collections import deque
import matplotlib.pyplot as plt

yolo = YOLO('yolov8s.pt')  # Load the pre-trained YOLOv8 small model



# Path to input video (webcam input in this case for real-time processing)
videoCap = cv2.VideoCapture(1)  # Use 0 for the default webcam

# Initialize output video capture and display parameters
frame_width = int(videoCap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(videoCap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = 30  # Typically 30fps for real-time video
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
output_video_path = 'output.mp4'
out = cv2.VideoWriter(output_video_path, fourcc, fps, (frame_width, frame_height))

# Critical objects to alert
critical_objects = ['person', 'car', 'bus', 'truck']

# Approximation for object size-to-distance (example values)
KNOWN_WIDTH = 0.5  # Average width of a person in meters
FOCAL_LENGTH = 800  # Focal length in pixels (calibrated for your camera)

# Delay control for alerts
last_alert_time = time.time()
alert_delay = 3  # Delay in seconds between voice alerts

# Queue for voice messages
voice_queue = deque()

# Initialize object description for the final message
object_description = "No objects detected."

# Estimate distance function
def estimate_distance(width_in_pixels):
    if width_in_pixels == 0:
        return float('inf')
    return (KNOWN_WIDTH * FOCAL_LENGTH) / width_in_pixels

# Categorize distance
def categorize_distance(distance):
    if distance < 2:
        return "very close"
    elif 2 <= distance < 5:
        return "nearby"
    elif 5 <= distance < 10:
        return "moderately far"
    else:
        return "far away"

# Process video frame by frame
while True:
    ret, frame = videoCap.read()
    if not ret:
        break

    # YOLO object detection
    results = yolo.track(frame, stream=True)
    detected_this_frame = []  # Track objects detected in the current frame

    # Reset object counts for each frame
    object_counts = {}

    for result in results:
        classes_names = result.names

        for box in result.boxes:
            if box.conf[0] > 0.4:  # Confidence threshold
                cls = int(box.cls[0])
                class_name = classes_names[cls]
                bbox = box.xywh[0]  # Bounding box (x_center, y_center, w, h)
                confidence = box.conf[0]

                # Calculate bounding box coordinates
                x_center, y_center, w, h = map(float, bbox)
                x1 = int(x_center - w / 2)
                y1 = int(y_center - h / 2)
                x2 = int(x_center + w / 2)
                y2 = int(y_center + h / 2)

                # Draw bounding box on the frame
                color = (0, 255, 0) if class_name not in critical_objects else (0, 0, 255)
                cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
                cv2.putText(frame, f'{class_name} {confidence:.2f}', (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, color, 2)

                # Estimate distance based on the object width in pixels
                distance = estimate_distance(float(w))
                distance_category = categorize_distance(distance)
                cv2.putText(frame, f'Distance: {distance:.2f}m', (x1, y2 + 20), cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2)

                # Count objects
                object_counts[class_name] = object_counts.get(class_name, 0) + 1

                # If critical object is detected, append alert message
                if class_name in critical_objects:
                    detected_this_frame.append(
                        f"Alert! A {class_name} is {distance_category}, approximately {round(distance, 1)} meters away."
                    )

    # Prepare message with all detected objects
    if object_counts:
        object_description = ', '.join([f"{count} {name}" for name, count in object_counts.items()])

    message = f"Objects detected: {object_description}"

    # Real-time audio feedback for detected objects
    if detected_this_frame and time.time() - last_alert_time > alert_delay:
        alert_message = detected_this_frame[0]
        tts = gTTS(alert_message, lang='en')

        # Save to a temporary file and play the audio message
        temp_file = tempfile.NamedTemporaryFile(delete=False, suffix=".mp3")
        tts.save(temp_file.name)
        temp_file.close()

        # Play the sound synchronously
        playsound(temp_file.name)
        os.unlink(temp_file.name)  # Clean up the temporary file

        last_alert_time = time.time()

    # Convert frame from BGR to RGB (because OpenCV uses BGR by default, but Matplotlib uses RGB)
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    # Display the frame using Matplotlib
    plt.imshow(frame_rgb)
    plt.axis('off')  # Hide axis
    plt.draw()  # Update the plot
    plt.pause(0.01)  # Pause briefly to allow for rendering (useful for live video)

    # Write the frame to output video file
    out.write(frame)

    # Exit the loop after a certain number of frames or conditions
    # For example, we can break after a certain time (e.g., 30 seconds)
    if time.time() - last_alert_time > 30:  # Adjust time to your needs
        break
        
# Release video resources and clean up
videoCap.release()
out.release()

# Safely handle destroyAllWindows()
try:
    cv2.destroyAllWindows()
except cv2.error:
    pass

# Final audio alert after processing
final_message = f"Detection complete. Objects detected: {object_description}"
tts = gTTS(final_message, lang='en')
final_temp_file = tempfile.NamedTemporaryFile(delete=False, suffix=".mp3")
tts.save(final_temp_file.name)
final_temp_file.close()

# Play the final audio alert synchronously
playsound(final_temp_file.name)
os.unlink(final_temp_file.name)  # Clean up the temporary file


In [ ]:
import cv2
from ultralytics import YOLO
from gtts import gTTS
from playsound import playsound
import tempfile
import os
import time
from collections import deque
import matplotlib.pyplot as plt

# Load the pre-trained YOLOv8 small model
yolo = YOLO('yolov8s.pt')  # You can change the model based on your preference

# Path to input video (webcam input in this case for real-time processing)
videoCap = cv2.VideoCapture(1)  # Use 0 for the default webcam

# Initialize output video capture and display parameters
frame_width = int(videoCap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(videoCap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = 30  # Typically 30fps for real-time video
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
output_video_path = 'output.mp4'
out = cv2.VideoWriter(output_video_path, fourcc, fps, (frame_width, frame_height))

# Critical objects to alert
critical_objects = ['person', 'car', 'bus', 'truck']

# Approximation for object size-to-distance (example values)
KNOWN_WIDTH = 0.5  # Average width of a person in meters
FOCAL_LENGTH = 800  # Focal length in pixels (calibrated for your camera)

# Delay control for alerts
last_alert_time = time.time()
alert_delay = 3  # Delay in seconds between voice alerts

# Queue for voice messages
voice_queue = deque()

# Initialize object description for the final message
object_description = "No objects detected."

# Estimate distance function
def estimate_distance(width_in_pixels):
    if width_in_pixels == 0:
        return float('inf')
    return (KNOWN_WIDTH * FOCAL_LENGTH) / width_in_pixels

# Categorize distance
def categorize_distance(distance):
    if distance < 2:
        return "very close"
    elif 2 <= distance < 5:
        return "nearby"
    elif 5 <= distance < 10:
        return "moderately far"
    else:
        return "far away"

# Process video frame by frame
while True:
    ret, frame = videoCap.read()
    if not ret:
        break

    # YOLO object detection
    results = yolo.track(frame, stream=True)
    detected_this_frame = []  # Track objects detected in the current frame

    # Reset object counts for each frame
    object_counts = {}

    for result in results:
        classes_names = result.names

        for box in result.boxes:
            if box.conf[0] > 0.4:  # Confidence threshold
                cls = int(box.cls[0])
                class_name = classes_names[cls]
                bbox = box.xywh[0]  # Bounding box (x_center, y_center, w, h)
                confidence = box.conf[0]

                # Calculate bounding box coordinates
                x_center, y_center, w, h = map(float, bbox)
                x1 = int(x_center - w / 2)
                y1 = int(y_center - h / 2)
                x2 = int(x_center + w / 2)
                y2 = int(y_center + h / 2)

                # Draw bounding box on the frame
                color = (0, 255, 0) if class_name not in critical_objects else (0, 0, 255)
                cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
                cv2.putText(frame, f'{class_name} {confidence:.2f}', (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, color, 2)

                # Estimate distance based on the object width in pixels
                distance = estimate_distance(float(w))
                distance_category = categorize_distance(distance)
                cv2.putText(frame, f'Distance: {distance:.2f}m', (x1, y2 + 20), cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2)

                # Count objects
                object_counts[class_name] = object_counts.get(class_name, 0) + 1

                # If critical object is detected, append alert message
                if class_name in critical_objects:
                    detected_this_frame.append(
                        f"Alert! A {class_name} is {distance_category}, approximately {round(distance, 1)} meters away."
                    )

    # Prepare message with all detected objects
    if object_counts:
        object_description = ', '.join([f"{count} {name}" for name, count in object_counts.items()])

    message = f"Objects detected: {object_description}"

    # Real-time audio feedback for detected objects
    if detected_this_frame and time.time() - last_alert_time > alert_delay:
        alert_message = detected_this_frame[0]
        tts = gTTS(alert_message, lang='en')

        # Save to a temporary file and play the audio message
        temp_file = tempfile.NamedTemporaryFile(delete=False, suffix=".mp3")
        tts.save(temp_file.name)
        temp_file.close()

        # Play the sound synchronously
        playsound(temp_file.name)
        os.unlink(temp_file.name)  # Clean up the temporary file

        last_alert_time = time.time()

    # Convert frame from BGR to RGB (because OpenCV uses BGR by default, but Matplotlib uses RGB)
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    # Display the frame using Matplotlib
    plt.imshow(frame_rgb)
    plt.axis('off')  # Hide axis
    plt.draw()  # Update the plot
    plt.pause(0.01)  # Pause briefly to allow for rendering (useful for live video)

    # Write the frame to output video file
    out.write(frame)

    # Exit the loop after a certain number of frames or conditions
    # For example, we can break after a certain time (e.g., 30 seconds)
    if time.time() - last_alert_time > 30:  # Adjust time to your needs
        break
        
# Release video resources and clean up
videoCap.release()
out.release()

# Safely handle destroyAllWindows()
try:
    cv2.destroyAllWindows()
except cv2.error:
    pass

# Final audio alert after processing
final_message = f"Detection complete. Objects detected: {object_description}"
tts = gTTS(final_message, lang='en')
final_temp_file = tempfile.NamedTemporaryFile(delete=False, suffix=".mp3")
tts.save(final_temp_file.name)
final_temp_file.close()

# Play the final audio alert synchronously
playsound(final_temp_file.name)
os.unlink(final_temp_file.name)  # Clean up the temporary file
